# Notebook 1 — Spectral Decomposition of the Averaging Operator

We build the averaging operator $A = \frac{1}{|S|}\sum_{s\in S}\rho(s)$ from the 18 face-turn generators of the Rubik's Cube and diagonalize it. The result: **6 distinct rational eigenvalues**, a spectral collapse from 228 dimensions to 6 layers.

## 1. Build the Operator

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from rime.cubieoperator import CubieSpectralOperator, eigenspaces
from rime.cubie import CubieMove
from rime.cubieworld import SlowDynamics
from rime import helpers
from rime.spectral_utils import joint_diag_sectors, compute_transport_kappa
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
op = CubieSpectralOperator.from_gens_dict(CubieMove.prim_moves)
layers = sorted(op._layers, reverse=True)
print(f'Operator built: {len(layers)} spectral layers, dims={[op._layer_dim(lam) for lam in layers]}')

## 2. The 18 Face-Turn Generators

`CubieMove.prim_moves` contains all 18 standard moves (R, R', R2, U, U', U2, F, F', F2, L, L', L2, D, D', D2, B, B', B2). Each is a group element acting on the 228-dim cubie representation.

In [ ]:
# Show all 18 generators
for key, (mv, rho, *_) in op.rho_moves.items():
    axis, side, direction = key
    face = {0: 'RL', 1: 'UD', 2: 'FB'}[axis][0 if side > 0 else 1]
    dir_str = {1: '', -1: "'", 2: '2'}[direction]
    print(f"  {face}{dir_str:2s}  axis={axis} side={side:+d} dir={direction:+d}")
    

## 3. Eigenvalues and Spectral Layers

In [ ]:
# Diagonalize A and show the six layers
evals = sorted(op._layers, reverse=True)
print(f"{'λ':>10} {'k':>3} {'dim':>5} {'label':>10}")
print("-" * 35)
for lam in evals:
    d = op._layer_dim(lam)
    k = round((1 - lam) * 9)
    label = f"V_{lam:.4f}"
    print(f"{lam:>10.6f} {k:>3} {d:>5} {label:>10}")
print(f"\nTotal: {sum(op._layer_dim(lam) for lam in evals)} = 228 ✓")

## 4. Spectral Tower

Each spectral layer $V_\lambda$ is an eigenspace of $A$. The layer dimensions vary dramatically — $V_{5/9}$ is the largest (106 dim), $V_{8/9}$ the smallest (2 dim).

In [ ]:
layers_sorted = sorted(op._layers, reverse=True)
dims = [op._layer_dim(lam) for lam in layers_sorted]
labels = [f'V_{lam:.4f}' for lam in layers_sorted]
colors = ['#1a1a2e', '#16213e', '#0f3460', '#533483', '#e94560', '#f5f5f5']

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
y_pos = range(len(layers_sorted))
ax.barh(y_pos, dims, color=colors, edgecolor='white', linewidth=1.5, height=0.6)
for i, (d, lam) in enumerate(zip(dims, layers_sorted)):
    k = round((1 - lam) * 9)
    ax.text(d + 2, i, f'$V_{{{k}/{9}}}$  ({d}d)', va='center', fontsize=12, fontweight='bold')
ax.set_yticks(y_pos)
ax.set_yticklabels([f'λ = {lam:.4f}' for lam in layers_sorted], fontsize=10)
ax.set_xlabel('Dimension', fontsize=12)
ax.set_title('Spectral Layers of the Averaging Operator $A_{18}$', fontsize=14)
ax.set_xlim(0, max(dims) + 50)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Block Structure

The 228-dim representation decomposes into four cubie-type blocks:
- **cp** (64): corner permutation
- **ep** (144): edge permutation
- **co** (8): corner orientation
- **eo** (12): edge orientation

Each spectral layer has a characteristic block support pattern.

In [ ]:
from rime.spectralstructure import SpectralStructure
ss = SpectralStructure.from_rho_moves(op.rho_moves)
block_projs = ss.block_projectors()
block_names = ['cp', 'ep', 'co', 'eo']
block_labels = ['CP (64)', 'EP (144)', 'CO (8)', 'EO (12)']

for lam in layers_sorted:
    P = op.eigenspace_projector(lam)
    weights = []
    for bn in block_names:
        Pb = block_projs[bn]
        w = np.linalg.norm(Pb @ P, 'fro')**2
        weights.append(w)
    weights = np.array(weights)
    weights /= weights.sum()
    bar = ''.join(f'{bn}:{w:.0%} ' for bn, w in zip(block_names, weights) if w > 0.01)
    print(f'  V_{lam:.4f} ({op._layer_dim(lam):>3}d):  {bar}')

## 6. The Rational Spectral Law

All six eigenvalues follow $\lambda = 1 - k/m$ with $m=9$ and $k \in \{0,1,2,3,4,6\}$. The value $k=5$ is **genuinely absent** — this gap is forced by the blockwise Bose-Mesner algebra structure.

The rationality does not require generator commutativity. It follows from **partition integrality**: the face-sum matrices have integer entries, and the per-face eigenspace traces are integers. See Paper I §6 for the proof.